# In-Class Activity: Build a Basic Data Pipeline — SOLUTIONS

**MATH 153 · Data Wrangling / Data Engineering**
Howard University · Department of Mathematics

---

### The situation

Founders Library records every book checkout and sends the circulation office **one file per day**.
Three days have arrived. The office wants a clean table they can query, plus a daily summary of
how many books went out by genre.

**Your job: build the ETL pipeline.** Same five components as the café example — only the data
is different.

| | |
|---|---|
| **Source** | `activity_data/checkouts_2027-03-0{2,3,4}.csv` and `activity_data/books.csv` |
| **Extract** | read the three daily files and stack them |
| **Transform** | fix `loan_days`, clean `patron_type`, join `books`, add a due date |
| **Load** | write to `library.db` and `checkout_summary.csv` |
| **Schedule** | (we'll just talk about this one) |

### How to work

Six tasks. Each has a cell with `# TODO` in it. Fill in the blanks and run the cell — each one
prints something you can check against the expected answer written underneath.

Work in pairs. Ask when you are stuck; everything you need is in the lecture notebook.

---
## Setup — run this first

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DATA = Path("activity_data")
print("files:", sorted(p.name for p in DATA.iterdir()))

files: ['books.csv', 'checkouts_2027-03-02.csv', 'checkouts_2027-03-03.csv', 'checkouts_2027-03-04.csv']


## Look before you build

Always do this first. Two things in this file need fixing — find them before you read on.

In [2]:
peek = pd.read_csv(DATA / "checkouts_2027-03-02.csv")
print(peek.shape)
peek.head()

(26, 5)


,checkout_id,checkout_date,book_id,patron_type,loan_days
0,C-7001,2027-03-02,BK-106,STAFF,14 days
1,C-7002,2027-03-02,BK-107,Faculty,7 days
2,C-7003,2027-03-02,BK-106,GRADUATE,28 days
3,C-7004,2027-03-02,BK-106,GRADUATE,21 days
4,C-7005,2027-03-02,BK-102,Staff,28 days


Did you spot them?

1. **`loan_days` is text**, not a number — it says `"14 days"`. You cannot add `"14 days"` to a date.
2. **`patron_type` is spelled inconsistently** — `STAFF`, `Faculty`, `graduate`, `  Undergrad  `.

Those two are your Transform step, exactly like `unit_price` and `item` in the café example.

In [3]:
books = pd.read_csv(DATA / "books.csv")
books

,book_id,title,genre,replacement_cost
0,BK-101,Introductory Statistics,Mathematics,89.00
1,BK-102,"A History of Washington, DC",History,42.50
2,BK-103,Organic Chemistry,Science,134.00
3,BK-104,Poems of the Harlem Era,Literature,28.75
4,BK-105,Linear Algebra Done Right,Mathematics,64.00
5,BK-106,Public Health Policy,Health,71.25
6,BK-107,Jazz and the American City,Music,38.00
7,BK-108,Databases: A First Course,Computing,95.50


This is the **reference table** — it tells you each book's genre and replacement cost.
You will join it on in Task 4.

---
## Task 1 — Extract  *(5 min)*

Write a function that reads all three daily files and stacks them into one DataFrame.

**Hints:** `DATA.glob("checkouts_2027-*.csv")` finds the files. `sorted()` puts them in date order.
`pd.concat(frames, ignore_index=True)` stacks them.

In [4]:
def extract(data_dir):
    """Read every daily checkout file and stack them into one DataFrame."""
    files = sorted(data_dir.glob("checkouts_2027-*.csv"))
    frames = [pd.read_csv(f) for f in files]
    return pd.concat(frames, ignore_index=True)


raw = extract(DATA)
print("rows:", len(raw))
raw.head(3)

rows: 82


,checkout_id,checkout_date,book_id,patron_type,loan_days
0,C-7001,2027-03-02,BK-106,STAFF,14 days
1,C-7002,2027-03-02,BK-107,Faculty,7 days
2,C-7003,2027-03-02,BK-106,GRADUATE,28 days


**Expected: 82 rows.**

Remember the rule from the lecture — Extract changes nothing. No cleaning here.

---
## Task 2 — Transform, step 1: make `loan_days` a number  *(5 min)*

`"14 days"` needs to become `14`.

**Hint:** `.str.replace(" days", "", regex=False)` then `.astype(int)`.

In [5]:
df = raw.copy()        # always work on a copy

df["loan_days"] = df["loan_days"].str.replace(" days", "", regex=False).astype(int)

print("before:", raw["loan_days"].iloc[0], "| after:", df["loan_days"].iloc[0])
print("dtype :", df["loan_days"].dtype)

before: 14 days | after: 14
dtype : int64


**Expected:** the dtype should be `int64`, not `object`.

Also convert the date while you are here — a date stored as text is just a string.

In [6]:
df["checkout_date"] = pd.to_datetime(df["checkout_date"], format="%Y-%m-%d")

print("dtype:", df["checkout_date"].dtype)

dtype: datetime64[us]


---
## Task 3 — Transform, step 2: clean `patron_type`  *(5 min)*

Four patron types are hiding behind many different spellings. Make them consistent.

**Hint:** the café example used `.str.strip()` followed by `.str.title()`.

In [7]:
before = df["patron_type"].nunique()

df["patron_type"] = df["patron_type"].str.strip().str.title()

print(f"spellings before: {before}")
print(f"types after     : {df['patron_type'].nunique()}")
print(sorted(df["patron_type"].unique()))

spellings before: 16
types after     : 4
['Faculty', 'Graduate', 'Staff', 'Undergrad']


**Expected: 16 spellings become 4** — `Faculty`, `Graduate`, `Staff`, `Undergrad`.

**Why does this step have to come before the join?** Write your answer in one sentence:

> *your answer here*

---
## Task 4 — Transform, step 3: join the books, add a due date  *(7 min)*

Join `books` so every checkout knows its genre, then work out when each book is due back.

**Hints:** join on `book_id` with `how="left"`. For the due date:
`df["checkout_date"] + pd.to_timedelta(df["loan_days"], unit="D")`.

In [8]:
df = df.merge(books, on="book_id", how="left")

print("rows after the join:", len(df), " <- should still be 82")
print("rows with no matching book:", df["genre"].isna().sum(), " <- should be 0")

rows after the join: 82  <- should still be 82
rows with no matching book: 0  <- should be 0


In [9]:
df["due_date"] = df["checkout_date"] + pd.to_timedelta(df["loan_days"], unit="D")

df[["checkout_id", "checkout_date", "title", "genre", "loan_days", "due_date"]].head()

,checkout_id,checkout_date,title,genre,loan_days,due_date
0,C-7001,2027-03-02,Public Health Policy,Health,14,2027-03-16
1,C-7002,2027-03-02,Jazz and the American City,Music,7,2027-03-09
2,C-7003,2027-03-02,Public Health Policy,Health,28,2027-03-30
3,C-7004,2027-03-02,Public Health Policy,Health,21,2027-03-23
4,C-7005,2027-03-02,"A History of Washington, DC",History,28,2027-03-30


**The two checks to make after every join** — and they are printed above, so look at them:

- the row count stayed at 82 (if it grew, `books.csv` has a duplicate `book_id`)
- nothing failed to match (if some did, your `book_id` values need cleaning too)

---
## Task 5 — Load  *(6 min)*

Build the daily summary, then write both outputs: a database table and a CSV.

In [10]:
# TODO: one row per (date, genre), with a count of checkouts and the average loan length
summary = (df
           .assign(day=df["checkout_date"].dt.date)
           .groupby(["day", "genre"], as_index=False)
           .agg(checkouts=("checkout_id", "size"), avg_loan_days=("loan_days", "mean"))
           .rename(columns={"day": "checkout_date"}))
summary["avg_loan_days"] = summary["avg_loan_days"].round(1)
summary.head(8)

,checkout_date,genre,checkouts,avg_loan_days
0,2027-03-02,Computing,1,21.0
1,2027-03-02,Health,4,21.0
2,2027-03-02,History,3,23.3
3,2027-03-02,Literature,3,18.7
4,2027-03-02,Mathematics,9,11.7
5,2027-03-02,Music,6,16.3
6,2027-03-03,Computing,3,9.3
7,2027-03-03,Health,2,17.5


In [11]:
def load(clean, summary, db_path="library.db", csv_path="checkout_summary.csv"):
    """Write the results where people will actually read them."""
    out = clean.copy()
    out["checkout_date"] = out["checkout_date"].dt.strftime("%Y-%m-%d")
    out["due_date"] = out["due_date"].dt.strftime("%Y-%m-%d")

    con = sqlite3.connect(db_path)
    out.to_sql("checkouts", con, if_exists="replace", index=False)
    con.close()

    summary.to_csv(csv_path, index=False)
    return len(out)


n = load(df, summary)
print(f"loaded {n} rows into library.db")

loaded 82 rows into library.db


**Expected: 82 rows loaded.**

---
## Task 6 — Put it all together, and protect it  *(8 min)*

Wrap E, T and L in one function. Add a log line for each step and **two assertions** between
Transform and Load.

In [12]:
def run_pipeline(data_dir=DATA):
    """Extract -> Transform -> Load for the library checkout feed."""
    books = pd.read_csv(data_dir / "books.csv")

    # --- E
    raw = extract(data_dir)
    print(f"  EXTRACT   {len(raw):>4} rows read")

    # --- T
    clean = raw.copy()
    clean["loan_days"] = clean["loan_days"].str.replace(" days", "", regex=False).astype(int)
    clean["checkout_date"] = pd.to_datetime(clean["checkout_date"], format="%Y-%m-%d")
    clean["patron_type"] = clean["patron_type"].str.strip().str.title()
    clean = clean.merge(books, on="book_id", how="left")
    clean["due_date"] = clean["checkout_date"] + pd.to_timedelta(clean["loan_days"], unit="D")
    print(f"  TRANSFORM {len(clean):>4} rows cleaned")

    # --- CHECK
    assert clean["genre"].notna().all(), "some checkouts did not match a book"
    assert (clean["loan_days"] >= 1).all(), "a loan is less than one day long"

    # --- L
    summary = (clean
               .assign(day=clean["checkout_date"].dt.date)
               .groupby(["day", "genre"], as_index=False)
               .agg(checkouts=("checkout_id", "size"), avg_loan_days=("loan_days", "mean"))
               .rename(columns={"day": "checkout_date"}))
    load(clean, summary)
    print(f"  LOAD      {len(clean):>4} rows written")
    return clean, summary


clean, summary = run_pipeline()

  EXTRACT     82 rows read
  TRANSFORM   82 rows cleaned
  LOAD        82 rows written


### Now run it twice

A pipeline you cannot run twice is a pipeline nobody can trust.

In [13]:
run_pipeline()
run_pipeline()

con = sqlite3.connect("library.db")
print("rows in the table:", pd.read_sql_query("SELECT COUNT(*) AS n FROM checkouts", con)["n"][0])
con.close()

  EXTRACT     82 rows read
  TRANSFORM   82 rows cleaned
  LOAD        82 rows written


  EXTRACT     82 rows read
  TRANSFORM   82 rows cleaned


  LOAD        82 rows written
rows in the table: 82


**Expected: still 82, not 164 or 246.**

If you got 246, your `load` used `if_exists="append"`. Change it to `"replace"` and run again.

---
## Bonus — ask the database a question  *(if you have time)*

Your data is in SQL now. Use it.

In [14]:
con = sqlite3.connect("library.db")

pd.read_sql_query("""
    SELECT genre, COUNT(*) AS checkouts
    FROM   checkouts
    GROUP  BY genre
    ORDER  BY checkouts DESC
""", con)

,genre,checkouts
0,Mathematics,20
1,Music,15
2,Science,10
3,Literature,10
4,Health,10
5,Computing,9
6,History,8


In [15]:
pd.read_sql_query("""
    SELECT patron_type, COUNT(*) AS checkouts
    FROM   checkouts
    GROUP  BY patron_type
    ORDER  BY checkouts DESC
""", con)

,patron_type,checkouts
0,Undergrad,28
1,Faculty,19
2,Staff,18
3,Graduate,17


In [16]:
con.close()
print("pipeline complete")

pipeline complete


---
## Before you leave — three questions to discuss

1. **Scheduling.** Your pipeline reads every file in the folder. Tomorrow a fourth file arrives.
   What happens when you run `run_pipeline()` again? Is that what you want?

2. **Where do the checks belong?** You put two assertions between Transform and Load. What would
   go wrong if you put them at the very end, after `load()` had already run?

3. **One thing you would add.** You have a working pipeline. Name one thing it still needs before
   you would be comfortable letting it run unattended every morning.

### What you built today

```
extract()  ->  transform  ->  assert  ->  load()
   82 rows      82 rows       2 checks     library.db
                                           checkout_summary.csv
```

Four functions, two checks, and a result somebody else can use. That is a data pipeline.